In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq libassimp-dev
!pip install -q --upgrade "hyperdrone[examples]"
!pip install -q foundation-policy==1.0.1

In [ ]:
import math
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from foundation_policy import Raptor

from hyperdrone import dynamics, render
from hyperdrone.examples.data import procthor_scene_path, x500_model_path

WIDTH, HEIGHT = 512, 512
STEPS, FPS = 180, 30
TARGET = np.array([-3.92, -5.67, 1.0], dtype=np.float32)
START = TARGET + np.array([.5, 0, 0], dtype=np.float32)

scene = render.load_scene(procthor_scene_path(), fidelity="high")
assets = render.AssetPool()
drone_asset = assets.add_assembly(render.load_assembly(x500_model_path(), fidelity="high"))

sim = dynamics.Sim(num_drones=1, model="x500", device="auto")
sim.reset(seed=0, sample_states=False)
sim.state["position"] = START[None]
policy = Raptor()
policy.reset()
previous_action = np.zeros((1, sim.action_dim), dtype=np.float32)

renderer = render.Renderer(
    width=WIDTH, height=HEIGHT, num_cameras=2, output="rgb", fidelity="high",
    num_overlays=1, max_overlay_instances=8, max_overlays_per_camera=1,
)
renderer.init(scene, assets)
# Both, external and onboard camera should see the drone model:
renderer.attach(0, 0)  # Attach overlay (0) to the onboard camera (0)
renderer.attach(1, 0)  # Attach overlay (0) to the external camera (1)
# Spawn the drone model in the first overlay (0)
placement = renderer.spawn(0, drone_asset, render.make_transform(position=START))
renderer.update()

pitch = .3
c, s = math.cos(pitch), math.sin(pitch)
MOUNT = np.array([[c, 0, s, .1], [0, 1, 0, 0], [-s, 0, c, .32]], dtype=np.float32)


In [ ]:
frames = []
for _ in range(STEPS):
    observation = sim.observe()
    observation[:, :3] -= TARGET  # Position error for the step response.
    action = policy.evaluate_step(np.concatenate([observation, previous_action], axis=1))
    sim.step(action)
    previous_action = action

    position = sim.state.numpy("position")[0]
    orientation = sim.state.numpy("orientation")[0]
    transform = render.make_transform(position=position, orientation_wxyz=orientation)
    renderer.set_transform(0, placement, transform)
    renderer.update()

    onboard = sim.camera_bases_numpy(mount=MOUNT, fov=math.radians(100), aspect=renderer.aspect)
    external = renderer.camera(
        position=TARGET + np.array([-1.2, -1.2, .7]),
        look_at=TARGET,
        fov=math.radians(55),
    ).reshape(1, 12)
    renderer.set_cameras(np.concatenate([onboard, external]))
    renderer.render("rgb")
    views = renderer.frame()[..., :3]
    frames.append(np.concatenate([views[0], views[1]], axis=1))

mp4_path = Path("hyperdrone_sim.mp4")
imageio.mimsave(mp4_path, frames, fps=FPS)
print(f"Saved {mp4_path} (Raptor onboard/self-occlusion | fixed external view)")

from IPython.display import Image, Video, display
display(Video(str(mp4_path), embed=True))

## The RL environment

`hyperdrone.env.MultiEnvironment` is the C++ `rl_tools` environment
(`hyperdrone::MultiEnvironment<World>`) behind the exact batch verbs the C++ training
targets use — reset / render / observe / step / rewards / terminated. All environment
semantics live on the C++ side; a seeded rollout here is bit-exact against the C++
verbs. The first construction JIT-compiles the configuration (cached afterwards).


In [ ]:
from pathlib import Path
from hyperdrone.env import EnvConfig, MultiEnvironment

scenes = Path("scenes")
scenes.mkdir(exist_ok=True)
scene_file = Path(procthor_scene_path())
if not (scenes / scene_file.name).exists():
    (scenes / scene_file.name).symlink_to(scene_file)

env = MultiEnvironment(scenes, config=EnvConfig(instances=16, cam_width=64, cam_height=64), seed=0)
mask = np.ones(env.total_instances, dtype=np.uint8)
no_reset = np.zeros(env.total_instances, dtype=np.uint8)
env.reset(mask)
env.render(mask)
rng = np.random.default_rng(0)
for _ in range(5):
    actions = rng.uniform(-1, 1, size=(env.total_instances, env.action_dim)).astype(np.float32)
    env.step(actions)
    env.render(no_reset)
print("rewards:", env.rewards()[:4], "terminated:", env.terminated()[:4])
print("observation layout:", env.observation_layout)

import matplotlib.pyplot as plt
frames = env.frames()
figure, axes = plt.subplots(2, 4, figsize=(12, 6))
for view, axis in enumerate(axes.ravel()):
    axis.imshow(frames[view])
    axis.axis("off")
plt.tight_layout()
env.close()
